# Reinforcement Learning

Assume the environment will generate a tuple at each time step $t$: $(s_t, a_t, r_t, s_{t+1})$, where:
- $s_t$: the state at time $t$
- $a_t$: the action taken at time $t$
- $r_t$: the reward received after taking action $a_t$
- $s_{t+1}$: the state at time $t+1$

DRL agent learns a policy $\pi(a|s)$ that maximizes the expected cumulative reward $R = \sum_{t=0}^{T} \gamma^t r_t$, where $\gamma$ is the discount factor.


One typical RL algorithm is Q learning, which learns the action-value function $Q(s, a)$ that estimates the expected cumulative reward for taking action $a$ in state $s$.

## Algorithm Categories

- Online RL: 
  - value-based: 
    - DQN
    - DDQN
  - policy-based:
    - REINFORCE
    - PPO
- Offline RL:
  - Batch Constrained Q-learning (BCQ)
  - Conservative Q-learning (CQL)


## Tabular Q-learning
The $Q(s, a)$ function is represented as a table, where each entry corresponds to a state-action pair. The agent updates the Q-values using the Bellman equation:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left( r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right)$$

where:
- $\alpha$ is the learning rate
- $r$ is the reward received after taking action $a$ in state $s$
- $\gamma$ is the discount factor
- $s'$ is the next state after taking action $a$

The agent explores the environment and updates the Q-table based on the observed transitions. Over time, the Q-values converge to the true action-value function, allowing the agent to make optimal decisions.


```python
# Initialize Q-table with zeros
Q[s, a] = 0 for all states s and actions a

for episode in range(num_episodes):
    s = env.reset()
    while not done:
        # epsilon-greedy action
        a = epsilon_greedy(Q[s, :])

        # step in environment
        s_next, r, done = env.step(a)

        # target
        y = r + gamma * max_a' Q[s_next, a']

        # update
        Q[s, a] = Q[s, a] + alpha * (y - Q[s, a])

        # move on
        s = s_next
```

## DQN

DQN approximates $Q(s, a; θ)$ using a neural network with parameters $θ$, and maintain a delayed target network with parameters $θ'$. 
Two networks are used to stabilize training. 
The training process involves the following steps:

1. Sample a mini-batch of transitions from the replay buffer
2. Compute the target Q-values using the target network
3. Update the policy network by minimizing the loss between predicted and target Q-values

```python
# Two networks: online Q and target Q
online_net = QNetwork()
target_net = QNetwork()
target_net.load_state_dict(online_net.state_dict())   # sync once

replay_buffer = []

for episode in range(num_episodes):
    s = env.reset()
    while not done:
        # epsilon-greedy action from online net
        a = epsilon_greedy(online_net(s))

        s_next, r, done = env.step(a)
        replay_buffer.append((s, a, r, s_next, done))

        # sample mini-batch
        batch = sample(replay_buffer, B)

        # compute targets
        with torch.no_grad():
            # target network gives max over next actions
            q_next = target_net(batch.s_next)             # (B, num_actions)
            max_q_next = q_next.max(dim=1)                # (B,)
            y = batch.r + gamma * (1 - batch.done) * max_q_next

        # prediction from online net
        q_pred = online_net(batch.s).gather(1, batch.a)   # (B,)

        # loss and update
        loss = mse(q_pred, y)
        loss.backward()
        optimizer.step()

        # every N steps, sync target network
        if step % N == 0:
            target_net.load_state_dict(online_net.state_dict())

        s = s_next
```

## DDQN

DQN has a problem of overestimation bias, because it uses target network to both select and evaluate actions.
DDQN refines this by using the online network to select the actions and use the target network to evaluate it.


```python
# same setup as DQN
online_net = QNetwork()
target_net = QNetwork()
target_net.load_state_dict(online_net.state_dict())

for episode in range(num_episodes):
    s = env.reset()
    while not done:
        a = epsilon_greedy(online_net(s))
        s_next, r, done = env.step(a)
        replay_buffer.append((s, a, r, s_next, done))

        batch = sample(replay_buffer, B)

        with torch.no_grad():
            # step 1: online net chooses argmax action
            q_next_online = online_net(batch.s_next)              # (B, num_actions)
            best_a = q_next_online.argmax(dim=1)                  # (B,)

            # step 2: target net evaluates that action
            q_next_target = target_net(batch.s_next)              # (B, num_actions)
            q_eval = q_next_target.gather(1, best_a)              # (B,)

            y = batch.r + gamma * (1 - batch.done) * q_eval

        q_pred = online_net(batch.s).gather(1, batch.a)           # (B,)

        loss = mse(q_pred, y)
        loss.backward()
        optimizer.step()

        if step % N == 0:
            target_net.load_state_dict(online_net.state_dict())

        s = s_next
```



# Offline Reinforcement Learning

In online RL (like DQN/DDQN), the agent collects new data, so its replay buffer always reflects the current policy distribution.
In offline RL:
- You can only train from $\mathcal{D}$
- If your algorithm picks actions that are out of distribution (not in $\mathcal{D}$), your Q-function may assign exaggerated values (extrapolation error).
- This leads to overestimation and divergence.

So offline RL methods must constrain the learned policy to remain close to the dataset’s action distribution, or otherwise regularize the Q-values.


General recipe for offline reinforcement learning

- Input
  - dataset $\mathcal{D} = {(s, a, r, s', d)}$, collected from human demonstration, a logging policy or others
- Procedure
  - define a function approximator
    - Q network $Q(s, a; \theta)$
    - sometimes also policy network $\pi(a|s; \phi)$
  - train Q-values from the dataset
    - for each tuple $(s, a, r, s', d) \in \mathcal{D}$, compute a TD target
    - need avoid overestimation
  - Policy extraction
    - From the Q-function, greedy or softmax
    - or train an explicit policy network constrainted by data
  - Evaluate policy
    - on held-out offline data, 

```python
# Input: dataset D = {(s, a, r, s_next, done)}

# Step 1: build networks
Q = QNetwork()
pi = PolicyNetwork()  # optional, depends on algorithm

# Step 2: train loop
for (s, a, r, s_next, done) in dataset_loader:
    
    with torch.no_grad():
        # Vanilla DDQN target (baseline)
        best_a = Q(s_next).argmax()
        target = r + gamma * (1 - done) * Q_target(s_next)[best_a]

        # More advanced (CQL, IQL...) modify this target

    # TD loss
    q_pred = Q(s).gather(1, a)
    loss = mse(q_pred, target)

    # Optionally add regularization (e.g. CQL)
    loss += conservative_penalty(Q, s, a)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


## BCQ (Batch Constrained Q-learning)
- train a generative model, e.g., VAE, to model dataset actions $\hat a \sim G(s)$
- constrain policy to choose only actions similar to $\hat a$
- target 

$$y = r + \gamma Q(s', \hat a, \theta^-), \hat a \sim G(s)$$

## IQL (Implicit Q-learning)

IN offline RL, the big problem is distribution shift:
- if you take $max_a' Q(s', a')$, you're evaluating actions that my bot be in the dataset
- this can lead to overestimation and divergence


IQL solves this by never explicitly taking a max over actions. Instead, 
- value function regression, e.g., expectile regression
- advantage-weighted behavior cloning 


Three networks are used:
- Q-function: $Q(s, a; \theta_q)$
- Value function: $V(s; \theta_v)$
- Policy: $\pi(a|s; \theta_{\pi})$


Three Steps for Training:
1. train Q function with standard TD learning
   
   $$L_Q = \mathbb{E}_{(s,a,r,s')} [(Q(s,a) - (r+\gamma V(s')))^2]$$

   target = reward + discounted value of next state (not max-Q)

2. train value function with expectile regression

    $$L_V = \mathbb{E}_{(s,a)}[L_2^{\tau}(Q(s,a) - V(s))]$$

    where the expectile loss is:

    $$L_2^{\tau}(x) = \begin{cases}
      (1-\tau) x^2 & \text{if } x < 0 \\
      \tau x^2 & \text{if } x \geq 0
    \end{cases}$$

    This pulls $V(s)$ toward a high quantile of Q-values

3. train policy with advantage-weighted regression

   $$L_{\pi} = \mathbb{E}_{(s,a)}[e^{\frac{A(s,a)}{\beta}}\log \pi(a|s)]$$

   where the advantage is defined as:

   $$A(s,a) = Q(s,a) - V(s)$$

   If $A(s, a)$ is high (action better than baseline value), dataset action gets more weight. If low, weight is small.


```python
# Networks
Q = QNetwork()
V = ValueNetwork()
pi = PolicyNetwork()

for epoch in range(num_epochs):
    for batch in dataset:
        s, a, r, s_next, done = batch

        # -------- Step 1: Q update --------
        with torch.no_grad():
            target = r + gamma * (1 - done) * V(s_next)
        q_pred = Q(s, a)
        loss_Q = mse(q_pred, target)

        # -------- Step 2: Value update --------
        with torch.no_grad():
            q_vals = Q(s, a)
            diff = q_vals - V(s)
        # expectile regression loss
        weight = torch.where(diff >= 0, tau, 1 - tau)
        loss_V = (weight * diff**2).mean()

        # -------- Step 3: Policy update --------
        with torch.no_grad():
            adv = Q(s, a) - V(s)
            weights = torch.exp(adv / beta).clamp(max=clip)  # stabilize

        log_prob = pi.log_prob(s, a)
        loss_pi = -(weights * log_prob).mean()

        # -------- Optimize --------
        optimize(Q, loss_Q)
        optimize(V, loss_V)
        optimize(pi, loss_pi)
